In [ ]:
"""Accounting methods

Generating P&L with different accounting methods (FIFO and LIFO)

Attributes
----------
cocoon
transaction configuration
derived portfolios
accounting methods
"""
## Import Libraries & Connect to LUSID

In [ ]:
# Import LUSID
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
import finbourne_sdk_utils.cocoon.cocoon as cocoon_tools
import lusid_sample_data as sd
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
import pandas as pd
import numpy as np
import os
import pytz
import printer as prettyprint
from datetime import datetime
import pylab
import matplotlib.pyplot as plt

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")
if secrets_path is None:
    secrets_path = os.path.join(os.path.dirname(os.getcwd()), "secrets.json")

api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook")

In [ ]:
%matplotlib inline
%pylab inline

# Set style to fivethirtyeight to create clean and clear looking graphs
plt.style.use('fivethirtyeight')

# Define a dictionary containing default plotting configurations
params = {
    'legend.fontsize': 'small',
    'figure.figsize': (15, 10),
    'axes.labelsize': 'small',
    'axes.titlesize':'medium',
    'xtick.labelsize':'small',
    'ytick.labelsize':'small'
}

pylab.rcParams.update(params)

## Import and Create Instruments

In [ ]:
uk_instrument_master = pd.read_csv("data/global-fund-UK-instrument-master.csv")
uk_instrument_master.head(n=20)

In [ ]:
instrument_properties_scope = 'InstrumentProperties005'

instrument_identifier_mapping = {
    "Figi": "figi",
    "Isin": "isin",
    "ClientInternal": "client_internal",
    "Ticker": "ticker"
}

instrument_mapping_required = {
    "name": "instrument_name"
}

instrument_mapping_optional = {}

responses = cocoon_tools.load_from_data_frame(
    api_factory=api_factory, 
    scope=instrument_properties_scope, 
    data_frame=uk_instrument_master, 
    mapping_required=instrument_mapping_required, 
    mapping_optional=instrument_mapping_optional, 
    file_type="instrument", 
    identifier_mapping=instrument_identifier_mapping, 
    property_columns=[
        "exchange_code", 
        "country_issue", 
        "market_sector",
        "security_type",
        "coupon",
        "s&p rating",
        "currency"])

prettyprint.instrument_response(responses["instruments"]["success"][0])

## Create a Portfolio with "FirstInFirstOut" (FIFO) Accounting Method

In [ ]:
portfolio_code = "UK12XC2"

portfolio = pd.DataFrame([
    {
        "code": portfolio_code,
        "display_name": "UK Equity Fund",
        "description": "UK Growth Fund Trading in UK Tradeable Equities",
        "base_currency": "GBP",
        "accounting_method": "FirstInFirstOut",
        "created": datetime.datetime(year=2010, day=1, month=1, tzinfo=pytz.UTC)
    }
])

portfolio.head()

In [ ]:
trading_scope = "AccountingDemoScope"

portfolio_mapping_required = {
    "code": "code",
    "display_name": "display_name",
    "base_currency": "base_currency",
    "accounting_method": "accounting_method"
}

portfolio_mapping_optional = {
    "description": "description",
    "created": "created"
}

responses = cocoon_tools.load_from_data_frame(
    api_factory=api_factory, 
    scope=trading_scope, 
    data_frame=portfolio, 
    mapping_required=portfolio_mapping_required, 
    mapping_optional=portfolio_mapping_optional, 
    file_type="portfolio")

prettyprint.portfolio_response(responses["portfolios"]["success"][0])

## Import and Create Transactions for September 2019

In [ ]:
transactions = pd.read_csv("data/accounting-demo-sample-transactions.csv")
transactions["cash_transactions"] = np.where(transactions["desc"] == "FundsIn", transactions["trade_currency"], np.nan) 
transactions.head(n=20)

In [ ]:
x = [datetime.date.fromisoformat(date.split("T")[0]) for date in transactions["trade_datetime"].values[1:]]
y = transactions["unit_price"].values[1:]
labels = [
    "-".join([
        transactions["desc"].values[i], 
        transactions["asset_name"].values[i]
    ]) for i in range(1, len(transactions["desc"].values))]

fig, ax = plt.subplots()
for x, y, label in zip(x,y,labels):
    if "Buy" in label:
        colour = "blue"
    else:
        colour = "red"

    ax.plot_date(x=x, y=y, c=colour)
    ax.annotate(label, (x, y), rotation=-5)

plt.xlabel('Trade Date', fontsize=18)
plt.ylabel('Unit Price (£)', fontsize=18)
fig.suptitle('Transactions for September', fontsize=20)
plt.show()

In [ ]:
transaction_identifier_mapping = {
    "Figi": "figi_identifier",
    "Isin": "isin_identifier",
    "ClientInternal": "internal_identifier",
    "Currency": "cash_transactions"
}

transaction_mapping_required = {
    "code": "fund_code",
    "transaction_id": "t_id",
    "type": "desc",
    "transaction_date": "trade_datetime",
    "settlement_date": "settlement_datetime",
    "units": "qty",
    "transaction_price.price": "unit_price",
    "transaction_price.type": "$Price",
    "total_consideration.amount": "total_trade_amount",
    "total_consideration.currency": "trade_currency"
}

transaction_mapping_optional = {
    "exchange_rate": "$1",
    "transaction_currency": "trade_currency"
}

responses = cocoon_tools.load_from_data_frame(
    api_factory=api_factory, 
    scope=trading_scope, 
    data_frame=transactions, 
    mapping_required=transaction_mapping_required, 
    mapping_optional=transaction_mapping_optional, 
    file_type="transactions", 
    identifier_mapping=transaction_identifier_mapping, 
    property_columns=[
        "exposure",
        "executor",
        "asset_name"
    ])

prettyprint.transactions_response(
    responses["transactions"]["success"][0], 
    trading_scope, 
    responses["transactions"]["success"][0].href.split("/")[7])

## List Transaction Types

In [ ]:
# Call LUSID to get your transaction type configuration
response = api_factory.build(lu.TransactionConfigurationApi).list_transaction_types()
# Pretty print the configuration
prettyprint.transaction_type_response(response, filters=["Buy", "Sell", "FundsIn"])

## Generate Holdings by Tax Lot At 30th September 2019

In [ ]:
response = api_factory.build(lu.TransactionPortfoliosApi).get_holdings(
    scope=trading_scope,
    code=portfolio_code,
    property_keys=[
        "Instrument/default/Name",
        "Instrument/default/Figi",
        f"Instrument/{instrument_properties_scope}/security_type"
    ],
    by_taxlots=True,
    effective_at=datetime.datetime(year=2019, month=9, day=30, hour=12, tzinfo=pytz.UTC).isoformat())

prettyprint.get_holdings_df(response)

## Generate P/L for September 2019

In [ ]:
response = api_factory.build(lu.TransactionPortfoliosApi).build_transactions(
    scope=trading_scope,
    code=portfolio_code,
    transaction_query_parameters=models.TransactionQueryParameters(
        start_date=datetime.datetime(year=2019, month=9, day=1, hour=0, tzinfo=pytz.UTC).isoformat(),
        end_date=datetime.datetime(year=2019, month=9, day=30, hour=0, tzinfo=pytz.UTC).isoformat(),
        query_mode="TradeDate"
        )
    )

profit_loss_fifo = prettyprint.output_transactions(response, trading_scope, portfolio_code)
profit_loss_fifo["accounting_method"] = "FIFO"
profit_loss_fifo

## Create Derived Portfolio with "LastInFirstOut" accounting method and Generate P/L

In [ ]:
try:
    response = api_factory.build(lu.PortfoliosApi).get_portfolio(
        scope=trading_scope,
        code=portfolio_code+"LIFO")
    
except ApiException as e:
    if e.status == 404:
        response = api_factory.build(lu.DerivedTransactionPortfoliosApi).create_derived_portfolio(
            scope=trading_scope,
            create_derived_transaction_portfolio_request=models.CreateDerivedTransactionPortfolioRequest(
                code=portfolio_code+"LIFO",
                display_name="LIFO Accounting treatment for the fund",
                parent_portfolio_id=models.ResourceId(
                    scope=trading_scope,
                    code=portfolio_code),
                accounting_method="LastInFirstOut",
                created=datetime.datetime(year=2010, month=1, day=1, tzinfo=pytz.UTC).isoformat()
            )
        )

prettyprint.portfolio_response(response)

In [ ]:
response = api_factory.build(lu.TransactionPortfoliosApi).build_transactions(
    scope=trading_scope,
    code=portfolio_code+"LIFO",
    transaction_query_parameters=models.TransactionQueryParameters(
        start_date=datetime.datetime(year=2019, month=9, day=1, hour=0, tzinfo=pytz.UTC).isoformat(),
        end_date=datetime.datetime(year=2019, month=9, day=30, hour=0, tzinfo=pytz.UTC).isoformat(),
        query_mode="TradeDate"
        )
    )

profit_loss_lifo = prettyprint.output_transactions(response, trading_scope, portfolio_code+"LIFO")
profit_loss_lifo["accounting_method"] = "LIFO"
profit_loss_lifo

In [ ]:
profit_loss_both = pd.concat([profit_loss_lifo, profit_loss_fifo]) \
                   .dropna(subset=["Realised Gain Loss", "Transaction ID"], axis=0) \
                   .sort_values("accounting_method")

x = np.arange(len(profit_loss_both["accounting_method"].unique()))
gain_loss_just_eat = profit_loss_both.loc[
    profit_loss_both["Transaction/AccountingDemoScope/exposure"] == "JustEat", "Realised Gain Loss"].unique()
gain_loss_bp = profit_loss_both.loc[
    profit_loss_both["Transaction/AccountingDemoScope/exposure"] == "BP", "Realised Gain Loss"].unique()
width = 0.35

fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, gain_loss_just_eat, width, label='JustEat')
rects2 = ax.bar(x + width/2, gain_loss_bp, width, label='BP')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Realised Gain/Loss (£)')
ax.set_title('Realised Gain/Loss by Accounting Method')
ax.set_xticks(x)
ax.set_xticklabels(["FirstInFirstOut", "LastInFirstOut"])
ax.legend()
plt.show()
profit_loss_both